# ERK-KTR Full FOV Stimulation Pipeline
This notebook showcases how to use the ERK-KTR full FOV stimulation pipeline. The pipeline is designed to simulate the full field of view (FOV) stimulation of a cells with the ERK-KTR biosensor. As it is a demo experiment, the pipeline runs on the demo hardware provided by MicroManager. 

### Import required libraries

In [1]:
import os
import time
from rtm_pymmcore.data_structures import Channel, StimTreatment
import rtm_pymmcore.utils as utils
from pprint import pprint
import pandas as pd

### Experimental Settings

In [2]:
# from rtm_pymmcore.microscope.MMDemo import MMDemo

# mic = MMDemo()
# mic.mmc.setChannelGroup("Channel")

from rtm_pymmcore.microscope.Jungfrau import Jungfrau

mic = Jungfrau()
mic.mmc.setChannelGroup("TTL_ERK")

In [3]:
## Configuration options - set experiment timing, storage and stimulation parameters
# General timing and frame counts:
# FIRST_FRAME_STIMULATION: first frame index (0-based) when stimulation logic can start.
FIRST_FRAME_STIMULATION = 10
# N_FRAMES: (optional) total number of frames for a single long experiment. We split into phases below instead.
# N_FRAMES = 340
# Instead of a single N_FRAMES, this notebook uses phase-specific frame counts:
N_FRAMES_PHASE_1 = 20  # number of timesteps in phase 1 (pre-drug)
N_FRAMES_PHASE_2 = 60  # number of timesteps in phase 2 (post-drug)

# If you want the notebook/script to wait before starting the experiment, set this (hours).
# Useful for scheduling: set to a fractional value (e.g. 0.5 for 30 minutes).
SLEEP_BEFORE_EXPERIMENT_START_in_H = 0

# Timing for acquisition: interval between timesteps (seconds) and approximate time per FOV (seconds).
# TIME_BETWEEN_TIMESTEPS: time between frames (across all FOVs).
TIME_BETWEEN_TIMESTEPS = 20  # seconds between timesteps
# TIME_PER_FOV: approximate time required to image one FOV (used for scheduling/estimates).
TIME_PER_FOV = 3.75  # seconds per FOV (camera + stage moves + overhead)

# Display / bookkeeping options:
# If True, add a column/group that stores the last stimulation exposure applied to each FOV (helpful for QC).
ADD_STIM_EXPOSURE_GROUP = False  # set to True to save per-FOV last-stim-exposure info
# When True, stim timepoints will be distributed evenly across the available timesteps rather than using explicit lists.
REGULAR_SPACING_BETWEEN_STIMULATIONS = False  # True -> evenly spaced stim timings; False -> use explicit stim_timestep lists

## Storage path for the experiment - change to your desired directory and experiment name.
base_path = "E:\\Alex"  # example: 'E:\Alex' (double-escaped for JSON/Notebook)
experiment_name = "Test8"
path = os.path.join(base_path, experiment_name)

## Channels: images to acquire each timestep.
# Each Channel(...) maps to a microscope channel name configured in Micro-Manager/your device.
# If exposure or power is omitted, the hardware default (set in the device/GUI) will be used.
# Examples: specify different exposures per channel, or add channels without exposures to use defaults.
channels = []
channels.append(
    Channel(name="mScarlet3", exposure=100)
)  # example: long exposure for nuclear marker
channels.append(Channel(name="mCitrine", exposure=100))  # example: ERK-KTR reporter


# Experimental condition(s): a list of labels assigned to FOVs.
# You can repeat or expand this list to match the number of FOVs.
condition = [
    "FGFR",
]
condition = condition * 12

# Example alternatives (uncomment to use):
# - Repeat each condition multiple times: condition = [cond for cond in condition for _ in range(repeats)]
# - Create a long condition vector: condition = ["optoFGFR_high"] * 24 + ["optoFGFR"] * 24

# If using wellplates, set how many FOVs per well. Set to None if not using wellplates.
n_fovs_per_well = None  ## number of FOVs per well; use None for free-FOV experiments

# Stimulation parameters for optogenetics. Define a list of StimTreatment objects per phase.
# Notes on StimTreatment fields:
# - treatment_name: human-readable label for the treatment
# - stim_timestep: a tuple/list of integers (timesteps) when stimulation should occur, OR a range/tuple.
#       Examples: (10,20,30), list(range(10,100)), (tuple(range(10,100,1)))
# - stim_exposure_list: either a single exposure value (applied to all listed timesteps) or a list/tuple of exposures matching the timesteps.
# - auto_repeat_stim_exposure: when True and a single exposure value is provided, it will be repeated across timesteps.
# - stim_power/stim_channel_name/stim_channel_group: map the stimulation settings to your device/channel group.
# - stim_channel_device_name/stim_channel_power_property_name: lower-level device settings used by the hardware API.


# stim_phase_2 = [
#     StimTreatment(
#         treatment_name="Sustained Phase 2 post Drug",
#         stim_timestep=(
#             tuple(range(30, 120, 1))
#         ),  # many ways to define timesteps: tuple, list, range, or single integer
#         stim_exposure_list=100,  # single value will be repeated if auto_repeat_stim_exposure=True
#         auto_repeat_stim_exposure=True,
#         stim_power=10,
#         stim_channel_name="CyanStim",
#         stim_channel_group="TTL_ERK",
#         stim_channel_device_name="Spectra",
#         stim_channel_power_property_name="Cyan_Level",
#     )
# ]  # a list of StimTreatment objects; when multiple are supplied they will be assigned across FOVs according to the utils.apply_stim_treatments_to_df_acquire logic

# Print the final stim schedule for confirmation before running. Helpful to verify exposures/timings.
# for stim_phase in [
#     stim_phase_1,
#     stim_phase_2,
# ]:
#     utils.print_stim_exposures_timesteps(stim_phase)

## Define the Tools that you are using for the experiment (segmentors, trackers, feature extractors, stimulator).
from rtm_pymmcore.stimulation.base_stimulation import StimWholeFOV
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy
from rtm_pymmcore.feature_extraction.erk_ktr import FE_ErkKtr
from rtm_pymmcore.feature_extraction.optocheck_fe import OptoCheckFE
from rtm_pymmcore.segmentation.cellpose_v4 import CellposeV4

segmentators = None

stimulator = None
feature_extractor = None
tracker = None
optocheck = None


from rtm_pymmcore.img_processing_pip import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_optocheck=optocheck,
)
mic.set_pipeline(pipeline=pipeline)

Directory E:\Alex\Test8\raw created 
Directory E:\Alex\Test8\tracks created 


### GUI - Napari Micromanager

#### Load GUI

In [4]:
### Base GUI ###
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc
viewer.window.add_dock_widget(mm_wdg)
data_mda_fovs = None
load_from_file = False

Please execute the following code block, if you would like to set a custom ROI (e.g. smaller illumination than the full field of view of the camera). Execute it after you have started the camera once in the GUI. 

In [ ]:
if mic.SET_ROI_REQUIRED:
    mic.mmc.clearROI()
    mic.mmc.setROI(mic.ROI_X, mic.ROI_Y, mic.ROI_WIDTH, mic.ROI_HEIGHT)

### Map Experiment to FOVs

### Use FOVs to generate dataframe for acquisition

In [6]:
fovs = utils.generate_fov_objects(mic, viewer=viewer)


df_acquire = utils.generate_df_acquire(
    fovs,
    n_frames=N_FRAMES_PHASE_1,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    phase_name="PreDrug",
    phase_id=0,
    condition=condition,
)
# df_acquire = utils.apply_stim_treatments_to_df_acquire(
#     df_acquire,
#     stim_phase_1,
#     condition,
#     n_fovs_per_well=n_fovs_per_well,
#     add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
#     regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
# )
df_acquire

Total Experiment Time: 0.10555555555555556h


c:\Users\Jungfrau\Documents\alandolt\code\rtm-pymmcore\rtm_pymmcore\utils.py:178: FutureWarning: The `_dock_widgets` property is private and should not be used in any plugin code. Please use the `dock_widgets` property instead.
  data_mda_fovs = viewer.window._dock_widgets["MDA"].widget().value().stage_positions


,fov_object,fov,fov_x,fov_y,fov_z,fov_name,timestep,time,channels,fname,cell_line,phase,phase_id
0,<rtm_pymmcore.data_structures.Fov object at 0x...,0,3386.9,22159.4,None,0,0,0.0,"({'name': 'mScarlet3', 'exposure': 100, 'group...",000_00_00000,FGFR,PreDrug,0
1,<rtm_pymmcore.data_structures.Fov object at 0x...,0,3386.9,22159.4,None,0,1,20.0,"({'name': 'mScarlet3', 'exposure': 100, 'group...",000_00_00001,FGFR,PreDrug,0
2,<rtm_pymmcore.data_structures.Fov object at 0x...,0,3386.9,22159.4,None,0,2,40.0,"({'name': 'mScarlet3', 'exposure': 100, 'group...",000_00_00002,FGFR,PreDrug,0
3,<rtm_pymmcore.data_structures.Fov object at 0x...,0,3386.9,22159.4,None,0,3,60.0,"({'name': 'mScarlet3', 'exposure': 100, 'group...",000_00_00003,FGFR,PreDrug,0
4,<rtm_pymmcore.data_structures.Fov object at 0x...,0,3386.9,22159.4,None,0,4,80.0,"({'name': 'mScarlet3', 'exposure': 100, 'group...",000_00_00004,FGFR,PreDrug,0
5,<rtm_pymmcore.data_structures.Fov object at 0x...,0,3386.9,22159.4,None,0,5,100.0,"({'name': 'mScarlet3', 'exposure': 100, 'group...",000_00_00005,FGFR,PreDrug,0
6,<rtm_pymmcore.data_structures.Fov object at 0x...,0,3386.9,22159.4,None,0,6,120.0,"({'name': 'mScarlet3', 'exposure': 100, 'group...",000_00_00006,FGFR,PreDrug,0
7,<rtm_pymmcore.data_structures.Fov object at 0x...,0,3386.9,22159.4,None,0,7,140.0,"({'name': 'mScarlet3', 'exposure': 100, 'group...",000_00_00007,FGFR,PreDrug,0
8,<rtm_pymmcore.data_structures.Fov object at 0x...,0,3386.9,22159.4,None,0,8,160.0,"({'name': 'mScarlet3', 'exposure': 100, 'group...",000_00_00008,FGFR,PreDrug,0
9,<rtm_pymmcore.data_structures.Fov object at 0x...,0,3386.9,22159.4,None,0,9,180.0,"({'name': 'mScarlet3', 'exposure': 100, 'group...",000_00_00009,FGFR,PreDrug,0


In [6]:
df_acquire.drop(columns=["fov_z"], inplace=True)
df_acquire

,fov_object,fov,fov_x,fov_y,fov_name,timestep,time,channels,fname,cell_line,phase,phase_id
0,<rtm_pymmcore.data_structures.Fov object at 0x...,0,4004.3,21335.1,0,0,0.0,"({'name': 'mScarlet3', 'exposure': 100, 'group...",000_00_00000,FGFR,PreDrug,0
1,<rtm_pymmcore.data_structures.Fov object at 0x...,0,4004.3,21335.1,0,1,20.0,"({'name': 'mScarlet3', 'exposure': 100, 'group...",000_00_00001,FGFR,PreDrug,0
2,<rtm_pymmcore.data_structures.Fov object at 0x...,0,4004.3,21335.1,0,2,40.0,"({'name': 'mScarlet3', 'exposure': 100, 'group...",000_00_00002,FGFR,PreDrug,0
3,<rtm_pymmcore.data_structures.Fov object at 0x...,0,4004.3,21335.1,0,3,60.0,"({'name': 'mScarlet3', 'exposure': 100, 'group...",000_00_00003,FGFR,PreDrug,0
4,<rtm_pymmcore.data_structures.Fov object at 0x...,0,4004.3,21335.1,0,4,80.0,"({'name': 'mScarlet3', 'exposure': 100, 'group...",000_00_00004,FGFR,PreDrug,0
...,...,...,...,...,...,...,...,...,...,...,...,...
135,<rtm_pymmcore.data_structures.Fov object at 0x...,6,3384.2,22158.5,6,15,700.0,"({'name': 'mScarlet3', 'exposure': 100, 'group...",006_00_00015,FGFR,PreDrug,0
136,<rtm_pymmcore.data_structures.Fov object at 0x...,6,3384.2,22158.5,6,16,720.0,"({'name': 'mScarlet3', 'exposure': 100, 'group...",006_00_00016,FGFR,PreDrug,0
137,<rtm_pymmcore.data_structures.Fov object at 0x...,6,3384.2,22158.5,6,17,740.0,"({'name': 'mScarlet3', 'exposure': 100, 'group...",006_00_00017,FGFR,PreDrug,0
138,<rtm_pymmcore.data_structures.Fov object at 0x...,6,3384.2,22158.5,6,18,760.0,"({'name': 'mScarlet3', 'exposure': 100, 'group...",006_00_00018,FGFR,PreDrug,0


### Run experiment

In [7]:
for _ in range(0, SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600):
    time.sleep(1)

try:
    mm_wdg._core_link.cleanup()

except:
    pass

mic.run_experiment(df_acquire)

In [ ]:
mic.run_experiment(df_acquire_2)

In [ ]:
mic.post_experiment()
time.sleep(10)

utils.generate_exp_data_from_tracks(path)

from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

### Function to re-connect link with GUI if manually broken

The following functions can be used to manually re-make the connection between the GUI and the running rtm-pymmcore script. However, normally you don't need to execute them. 

In [8]:
### Manually reconnect pymmcore with napari-micromanager
from napari_micromanager._core_link import CoreViewerLink

mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

The following code block can be used to manually break the connection between GUI and Jupyter Notebook:


In [ ]:
### Break connection
# mm_wdg._core_link.cleanup()